In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import mean_absolute_error, classification_report, confusion_matrix, accuracy_score
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score

In [ ]:
# Task 1: Write your code here:

Q1_path = os.path.join(path, 'Q1_data.csv')
df_foodDel = pd.read_csv(Q1_path)

In [ ]:
# Task 2: Write your code here:

print(f"shape: {df_foodDel.shape}")
df_foodDel.head()

In [ ]:
# Task 3: Write your code here:

df_foodDel.info()

In [ ]:
# Task 4: Write your code here:

df_foodDel.describe()

In [ ]:
# Task 5: Write your code here:

print(f"Delivery_Time: {df_foodDel['Delivery_Time'].sum()}")

In [ ]:
# Task 1: Write your code here:

df_foodDel = df_foodDel.drop(columns=['Order_ID'])

df_foodDel

In [ ]:
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df_foodDel)

In [ ]:
# Task 2: Write your code here:

#missing values = [Weather, Traffic_Level, Time_of_Day, Courier_Experience_yrs, Delivery_Time]
df_foodDel['Weather'] = df_foodDel['Weather'].fillna('Clear')
df_foodDel['Traffic_Level'] = df_foodDel['Traffic_Level'].fillna('Low')
df_foodDel['Courier_Experience_yrs'] = df_foodDel['Courier_Experience_yrs'].fillna('0')
df_foodDel['Delivery_Time'] = df_foodDel['Delivery_Time'].fillna('0')

df_foodDel

In [ ]:
# Task 3: Write your code here:

def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_foodDel)

In [ ]:
# Task 4: Write your code here:
categorical_cols = df_foodDel.select_dtypes(include=["object"]).columns

for col in categorical_cols:
    le = LabelEncoder()
    df_foodDel[col] = le.fit_transform(df_foodDel[col].astype(str))

df_foodDel

In [ ]:
# Task 5: Write your code here:

numerical_cols =  df_foodDel.select_dtypes(include=["number"]).columns.drop("Delivery_Time")

scaler = StandardScaler()

# TODO: Apply fit_transform to scale the numerical columns
df_foodDel[numerical_cols] = scaler.fit_transform(df_foodDel[numerical_cols])

df_foodDel.head()

In [ ]:
# Task 6: Write your code here:

#we will use check_target_imbalance function to check automatic for saving time
#it will give us how much by % the data appear for seeing if it's balance or not

def check_target_imbalance(df, target_column):
  print("Target Distribution:")
  print(df[target_column].value_counts(normalize=True))
  sns.countplot(x=df[target_column])
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(df_foodDel, "Delivery_Time")

#so, the data is balance because there is no Delivery_Time that take like 60% or heigher

In [ ]:
# Task 1: Write your code here:

X = df_foodDel.drop('Delivery_Time', axis = 1).astype(float)
y = df_foodDel['Delivery_Time'].astype(float)

In [ ]:
# Task 2,3,4,5: Write your code here:

# Task 2
#kf = KFold(n_splits= 5, shuffle=True, random_state=42)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"time in Train: {X_train.shape}, Test: {X_test.shape}")
print(f"time in train: {y_train.sum()}, in test: {y_test.sum()}")

X_scaled = scaler.fit_transform(X)
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Task 3
model = RandomForestClassifier(n_estimators=100, max_depth=15,
                               class_weight='balanced', random_state=42)
model.fit(X_train_scaled, y_train)
print("Model trained!")

# Task 4
y_pred = model.predict(X_test)

mae_scores = []

mae_scores.append(mean_absolute_error(y_test, y_pred))

# Task 5

for train_idx, test_idx in kf.split(X_scaled):
    X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

model.fit(X_train, y_train)

print(f"MAE : {np.mean(mae_scores):.2f}")

In [ ]:
# Task 1: Write your code here:

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: